# Travel Agent API Client Demo

This notebook demonstrates how to interact with the Travel Agent API system step by step.

## Prerequisites
- Host Agent running on port 12000 (`python main.py` in app folder)
- (Optional) Trip Agent running (`python main.py` in trip_agent folder

In [27]:
# Import required libraries
import requests
import json
import time
from datetime import datetime, timedelta
import uuid

# Configuration
HOST_AGENT_URL = "http://localhost:12000"
TRIP_AGENT_URL = "localhost:10000"  # Adjust port if different
APP_NAME = "travel_demo"
USER_ID = "demo_user"

# Helper function to pretty print JSON responses
def print_response(response):
    if response.status_code == 200:
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error {response.status_code}: {response.text}")


## Step 1: Initialize or Register Agents

Choose one of the following options:


In [28]:
# Option A: Register Trip Agent (if running)

response = requests.post(
    f"{HOST_AGENT_URL}/agent/register",
    params={
        "path": TRIP_AGENT_URL,
        "app_name": APP_NAME,
        "fixed_prompt": True or False
    }
)
print("Registering Trip Agent:")
print_response(response)


Registering Trip Agent:
{
  "status": "success"
}


In [ ]:
# Option B: Initialize Host Agent in Standalone Mode
# Use this if NOT running Trip Agent

response = requests.post(
    f"{HOST_AGENT_URL}/agent/initialize",
    params={"app_name": APP_NAME}
)
print("Initializing Host Agent:")
print_response(response)


## Step 2: Create a Conversation

Every interaction needs a conversation context. Let's create one:


In [29]:
# Create a new conversation
response = requests.post(
    f"{HOST_AGENT_URL}/conversation/create",
    params={
        "app_name": APP_NAME,
        "user_id": USER_ID
    }
)

print("Creating conversation:")
print_response(response)

if response.status_code == 200:
    conversation_id = response.json()["result"]["conversation_id"]
    print(f"\nConversation ID: {conversation_id}")
else:
    print("Failed to create conversation!")
    conversation_id = None


Creating conversation:
{
  "jsonrpc": "2.0",
  "id": "7485201cd48a42dea7b24ecbd17a7494",
  "result": {
    "conversation_id": "02df65f2-5841-4658-a858-e750df63fa1c",
    "is_active": true,
    "name": "",
    "task_ids": []
  },
  "error": null
}

Conversation ID: 02df65f2-5841-4658-a858-e750df63fa1c


## Step 3: Send a Travel Planning Request

Now let's send a structured travel planning message with all required metadata:


In [ ]:
message_data = {
    "role": "user",
    "parts": [
        {
            "type": "text",
            "text": "what ever message here"
        }
    ],
    "metadata": {
        "conversation_id": conversation_id,
        "conversation_name": "what ever name here",
        "context_id": str(uuid.uuid4()),
        "user_id": USER_ID,
        "app_name": APP_NAME,
    }
}

response = requests.post(
    f"{HOST_AGENT_URL}/message/send/sse",
    json=message_data
)

## Step 4: Wait for Processing

The system needs time to process your request and generate travel recommendations. Let's wait a bit:


In [31]:
print_response(response)
if response.status_code == 200:
    message_response = response.json()
else:
    print("Failed to send message!")


{
  "jsonrpc": "2.0",
  "id": "d13886d3ffda4b9cbd9a34a0bd18b480",
  "result": {
    "message_id": "372c1bd5-6c7a-46e3-b6bb-d354f0c9a14f",
    "conversation_id": "02df65f2-5841-4658-a858-e750df63fa1c",
    "context_id": "0c69c3fa-0c23-4fc1-a8ef-e344edd198b5",
    "response": {
      "role": "agent",
      "parts": [
        {
          "type": "text",
          "text": "Here's a summary of your planned trip to Japan:\n\n*   **Highlights:** Explore Tokyo's iconic landmarks, experience the serene beauty of Hakone, and immerse yourselves in the vibrant culture of Akihabara and Asakusa. Enjoy local cuisine, from street food to fine dining.\n*   **Key Experiences:** Visits to Meiji Jingu Shrine, Tokyo National Museum, a day trip to Hakone, Tsukiji Outer Market, Senso-ji Temple, and Akihabara.\n*   **Recommendations:** Consider purchasing a Japan Rail Pass if you plan to travel extensively outside of Tokyo. Book accommodations and popular activities in advance. Be sure to try local specialtie

## Step 5: Retrieve Travel Planning Results

The results are divided into 4 parts:
1. **Notes/Summary** - General travel advice (from the message response)
2. **Flights** - Flight recommendations
3. **Hotels** - Hotel recommendations  
4. **Activities** - Activity suggestions


In [32]:
# 1. Get Flight Information
response = requests.get(
    f"{HOST_AGENT_URL}/trip/flights",
    params={"conversation_id": conversation_id}
)
print("=== FLIGHT RECOMMENDATIONS ===")
if response.status_code == 200:
    data = response.json()
    flights = data.get("result", [])
    if flights:
        for i, flight in enumerate(flights, 1):
            print(f"\nFlight {i}:")
            print(f"  Route: {flight.get('origin', 'N/A')} → {flight.get('destination', 'N/A')}")
            print(f"  Departure: {flight.get('departure_time', 'N/A')}")
            print(f"  Arrival: {flight.get('arrival_time', 'N/A')}")
            print(f"  Duration: {flight.get('total_duration_minutes', 'N/A')} minutes")
            print(f"  Price: ${flight.get('price', 'N/A')} (for {flight.get('adults', 1)} adults)")
            print(f"  Type: {flight.get('flight_type', 'N/A')}")
            print(f"  Airport: {flight.get('departure_airport', 'N/A')}")
            print(f"  Legroom: {flight.get('legroom', 'N/A')}")
            
            # Show extensions/amenities
            extensions = flight.get('extensions', [])
            if extensions:
                print(f"  Features: {', '.join(extensions)}")
                
            # Show if this is the best option
            if flight.get('is_best', False):
                print(f"  ⭐ BEST OPTION")

            # Show delay info
            if not flight.get('often_delayed', True):
                print(f"  ✅ Low delay risk")
    else:
        print("No flight data available yet.")
else:
    print(f"Error retrieving flights: {response.text}")

=== FLIGHT RECOMMENDATIONS ===

Flight 1:
  Route: HAN → NRT
  Departure: 2025-06-25T07:55:00
  Arrival: 2025-06-25T15:30:00
  Duration: 335 minutes
  Price: $249.0 (for 3 adults)
  Type: Round trip
  Airport: Noi Bai International Airport
  Legroom: 28 in
  Features: Below average legroom (28 in), Carbon emissions estimate: 293 kg
  ⭐ BEST OPTION
  ✅ Low delay risk

Flight 2:
  Route: HAN → NRT
  Departure: 2025-06-25T00:30:00
  Arrival: 2025-06-25T08:00:00
  Duration: 330 minutes
  Price: $285.0 (for 3 adults)
  Type: Round trip
  Airport: Noi Bai International Airport
  Legroom: 28 in
  Features: Below average legroom (28 in), Carbon emissions estimate: 293 kg
  ✅ Low delay risk

Flight 3:
  Route: HAN → NRT
  Departure: 2025-06-25T23:35:00
  Arrival: 2025-06-26T06:45:00
  Duration: 310 minutes
  Price: $397.0 (for 3 adults)
  Type: Round trip
  Airport: Noi Bai International Airport
  Legroom: 34 in
  Features: Above average legroom (34 in), Wi-Fi for a fee, In-seat power & USB out

In [34]:
# 2. Get Hotel Information
response = requests.get(
    f"{HOST_AGENT_URL}/trip/hotels",
    params={"conversation_id": conversation_id}
)
print("\n=== HOTEL RECOMMENDATIONS ===")
if response.status_code == 200:
    data = response.json()
    hotels = data.get("result", [])
    if hotels:
        for i, hotel in enumerate(hotels, 1):
            # Show if this is the best option
            print(f"\nHotel {i}: {hotel.get('name', 'N/A')}")
            if hotel.get('is_best', False):
                print(f"  ⭐ BEST OPTION")
            print(f"  Type: {hotel.get('hotel_type', 'N/A').title()}")
            print(f"  Stars: {'⭐' * hotel.get('hotel_class_stars', 0)} ({hotel.get('hotel_class_stars', 'N/A')} stars)")
            print(f"  Check-in: {hotel.get('check_in_date', 'N/A')[:10]}")  # Show only date part
            print(f"  Check-out: {hotel.get('check_out_date', 'N/A')[:10]}")
            print(f"  Price/Night: ${hotel.get('rate_per_night', 'N/A')}")
            print(f"  Guests: {hotel.get('adults', 0)} adults, {hotel.get('children', 0)} children")
            
            # Ratings
            overall_rating = hotel.get('overall_rating', 0)
            location_rating = hotel.get('location_rating', 0)
            review_count = hotel.get('review_count', 0)
            print(f"  Overall Rating: {overall_rating}/5 ({review_count:,} reviews)")
            print(f"  Location Rating: {location_rating}/5")
            
            # Amenities
            amenities = hotel.get('amenities', [])
            if amenities:
                print(f"  Amenities: {', '.join(amenities)}")
            
            # Nearby places
            nearby_places = hotel.get('nearby_places', [])
            if nearby_places:
                nearby_info = []
                for place in nearby_places[:3]:  # Show first 3 nearby places
                    nearby_info.append(f"{place.get('name', 'Unknown')} ({place.get('distance', 'N/A')})")
                print(f"  Nearby: {', '.join(nearby_info)}")
            
            # Deal information
            deal = hotel.get('deal', '')
            deal_desc = hotel.get('deal_description', '')
            if deal or deal_desc:
                print(f"  Deal: {deal} {deal_desc}".strip())
                
    else:
        print("No hotel data available yet.")
else:
    print(f"Error retrieving hotels: {response.text}")


=== HOTEL RECOMMENDATIONS ===

Hotel 1: APA Hotel Asakusa Tawaramachi-Ekimae
  ⭐ BEST OPTION
  Type: Hotel
  Stars: ⭐⭐⭐ (3 stars)
  Check-in: 2025-06-25
  Check-out: 2025-06-30
  Price/Night: $77.0
  Guests: 3 adults, 0 children
  Overall Rating: 4.1/5 (4,014 reviews)
  Location Rating: 4.8/5
  Amenities: Public bath, Open air bath, Free Wi-Fi, Parking ($), Restaurant, Child-friendly
  Nearby: Sensō-ji (7 min), Tawaramachi Station (2 min), Haneda Airport (29 min)

Hotel 2: APA Hotel & Resort Ryogoku Eki Tower
  Type: Hotel
  Stars: ⭐⭐⭐ (3 stars)
  Check-in: 2025-06-25
  Check-out: 2025-06-30
  Price/Night: $75.0
  Guests: 3 adults, 0 children
  Overall Rating: 4.1/5 (3,403 reviews)
  Location Rating: 4.3/5
  Amenities: Public bath, Onsen, Open air bath, Free Wi-Fi, Parking ($), Outdoor pool, Fitness centre, Restaurant, Accessible
  Nearby: Sensō-ji (7 min), Ryogoku Sta. (4 min), Haneda Airport (30 min)

Hotel 3: Shinjuku Granbell Hotel
  Type: Hotel
  Stars: ⭐⭐⭐⭐ (4 stars)
  Check-in:

In [35]:
# 3. Get Activity Recommendations
response = requests.get(
    f"{HOST_AGENT_URL}/trip/activities",
    params={"conversation_id": conversation_id}
)
print("\n=== ACTIVITY RECOMMENDATIONS ===")
if response.status_code == 200:
    response_data = response.json()
    activities = response_data.get("result", [])
    if activities:
        # Sort activities by day and time for better display
        activities_sorted = sorted(activities, key=lambda x: (x.get('day', ''), x.get('time', '')))
        
        for activity in activities_sorted:
            print(f"\nDay: {activity.get('day', 'N/A')}")
            print(f"  Time: {activity.get('time', 'N/A')}")
            print(f"  Activity: {activity.get('activity', 'N/A')}")
            print(f"  Category: {activity.get('category', 'N/A')}")
            print(f"  Duration: {activity.get('duration', 'N/A')} minutes")
            print(f"  Price: ${activity.get('price', 'N/A')} {activity.get('currency', 'USD')}")
            print(f"  Location: {activity.get('location', 'N/A')}")
            print(f"  Description: {activity.get('detail_description', 'N/A')}")
            print(f"  Status: {activity.get('booking_status', 'N/A')}")
            if activity.get('activity_metadata'):
                print(f"  Notes: {activity.get('activity_metadata', '')}")
    else:
        print("No activity data available yet.")
else:
    print(f"Error retrieving activities: {response.text}")


=== ACTIVITY RECOMMENDATIONS ===

Day: day1
  Time: 08:00:00
  Activity: Departure from Hanoi
  Category: travel
  Duration: 180 minutes
  Price: $0.0 USD
  Location: Noi Bai International Airport (HAN)
  Description: Transfer to Noi Bai International Airport (HAN) for your flight to Tokyo, Japan.
  Status: pending
  Notes: Allow ample time for airport procedures. Confirm flight details.

Day: day1
  Time: 11:00:00
  Activity: Flight to Tokyo
  Category: travel
  Duration: 360 minutes
  Price: $500.0 USD
  Location: Flight HAN to NRT
  Description: Fly from Hanoi (HAN) to Narita International Airport (NRT), Tokyo.
  Status: pending
  Notes: Flight duration approximately 6 hours. Check baggage allowance.

Day: day1
  Time: 17:00:00
  Activity: Arrival in Tokyo & Transfer to Hotel
  Category: travel
  Duration: 120 minutes
  Price: $50.0 USD
  Location: Shinjuku, Tokyo
  Description: Arrive at Narita International Airport (NRT). Clear customs and immigration. Transfer to your hotel in S

## Additional Useful Endpoints

Here are some other endpoints you might find useful:


In [36]:
# List all messages in the conversation
response = requests.get(
    f"{HOST_AGENT_URL}/api/message/list",
    params={"conversation_id": conversation_id}
)

print("=== CONVERSATION MESSAGES ===")
if response.status_code == 200:
    messages = response.json()["result"]
    print(f"Total messages: {len(messages)}")
    for i, msg in enumerate(messages[-3:]):  # Show last 3 messages
        print(f"\nMessage {i+1}:")
        print(f"  Role: {msg.get('role', 'N/A')}")
        # Extract text content if available
        if 'parts' in msg and msg['parts']:
            for part in msg['parts']:
                if part.get('type') == 'text':
                    text = part.get('text', '')[:100] + '...' if len(part.get('text', '')) > 100 else part.get('text', '')
                    print(f"  Content: {text}")
else:
    print(f"Error: {response.text}")


=== CONVERSATION MESSAGES ===
Total messages: 2

Message 1:
  Role: user
  Content: **User Request:**
I want to plan a trip from Ha Noi to Japan with 2 of my friends, budget 3k USD eac...

Message 2:
  Role: agent
  Content: Here's a summary of your planned trip to Japan:

*   **Highlights:** Explore Tokyo's iconic landmark...


In [37]:
# List all conversations for the user
response = requests.get(
    f"{HOST_AGENT_URL}/api/conversation/list",
    params={
        "app_name": APP_NAME,
        "user_id": USER_ID
    }
)

print("\n=== USER CONVERSATIONS ===")
if response.status_code == 200:
    conversations = response.json()["result"]
    print(f"Total conversations: {len(conversations)}")
    for conv in conversations:
        print(f"\nConversation ID: {conv.get('conversation_id', 'N/A')}")
        print(f"  Name: {conv.get('name', 'N/A') or 'Unnamed'}")
        print(f"  Active: {conv.get('is_active', False)}")
else:
    print(f"Error: {response.text}")



=== USER CONVERSATIONS ===
Total conversations: 5

Conversation ID: 02df65f2-5841-4658-a858-e750df63fa1c
  Name: Unnamed
  Active: True

Conversation ID: e34ab775-d959-43fe-a48d-2859f5366fd2
  Name: Unnamed
  Active: True

Conversation ID: cf4ca221-eb4d-4cde-b5b4-ab47fcd3d5c3
  Name: Unnamed
  Active: True

Conversation ID: a04d6db5-c3d9-4ebd-aa9c-3c76dd255548
  Name: Unnamed
  Active: True

Conversation ID: 737de7fa-330c-4652-9e04-e63e479235ab
  Name: Unnamed
  Active: True


## Cleanup (Optional)

If you want to delete the conversation and all its data:


In [ ]:
# Delete the conversation (uncomment to run)
# response = requests.delete(
#     f"{HOST_AGENT_URL}/conversation/delete",
#     params={"conversation_id": conversation_id}
# )
# 
# if response.status_code == 200:
#     print(f"Successfully deleted conversation {conversation_id}")
# else:
#     print(f"Error deleting conversation: {response.text}")


## Summary

This notebook demonstrated the complete workflow for using the Travel Agent API:

1. **Initialize** the system (with or without Trip Agent)
2. **Create a conversation** to get a unique conversation_id
3. **Send a travel request** with all required metadata fields
4. **Wait** for processing to complete
5. **Retrieve results** from different endpoints:
   - `/trip/flights` - Flight recommendations
   - `/trip/hotels` - Hotel suggestions
   - `/trip/activities` - Activity ideas
   - Notes/summary from the message response

### Key Points to Remember:
- Always include all required metadata fields when sending messages
- The conversation_id is crucial for all subsequent API calls
- Results may take a few seconds to generate
- Each result type (flights, hotels, activities) has its own endpoint

### Error Handling:
- Check response status codes
- Validate dates (must be future dates, start < end)
- Ensure budget > 0 and travelers >= 1
